# MRCD src function implementation notebook
Notebook moi de test cac ham trong src theo format markdown + code (actual vs expected), su dung real SLM va real LLM, va mo phong MRCD round updates.

## Kaggle Setup
Cell setup duoc giu theo format dang dung trong notebook cu.

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

login(token=user_secrets.get_secret("HF_TOKEN"))
print("login")

In [ ]:
import os

GITHUB_REPO = "https://github.com/Chinh-de/Fake-news-detection.git"
REPO_NAME = "Fake-news-detection"

if not os.path.exists(REPO_NAME):
    !git clone {GITHUB_REPO}
%cd {REPO_NAME}

In [ ]:
!pip install -r requirements.txt --quiet

In [ ]:
from pathlib import Path
import json
import random
import sys

import numpy as np
import pandas as pd

REPO_ROOT = Path('/kaggle/working/Fake-news-detection')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import (
    MODEL_PATH,
    LLM_MODEL_NAME,
    TRAIN_CSV,
    VAL_CSV,
    TEST_CSV,
    KNOWLEDGE_MODE,
    TOP_K_DEMOS,
    FACT_TOP_K,
)
from src.utils import set_seed, preprocess_text, clean_query, truncate_text
from src.labels import generate_demo_label, to_clean_demo_label, parse_llm_label
from src.prompts import (
    build_dual_extraction_prompt,
    build_entity_extraction_prompt,
    build_classification_prompt,
)
from src.retrieval.demo_retrieval import load_news_corpus, search_news, retrieve_demonstrations
from src.retrieval.knowledge_agent import (
    query_wikipedia,
    extract_wiki_knowledge_from_entities,
    format_verified_reports,
    format_entity_definitions,
    build_knowledge_wiki_only,
    build_knowledge_full,
    build_knowledge_bundle,
    get_cached_knowledge_bundle_local,
)
from src.retrieval.knowledge_retrieval import (
    analyze_claim_entities_and_query,
    build_trusted_domain_query,
    scrape_full_article,
    _crawl_single_result,
    crawl_results_parallel,
    chunk_text_by_sentences,
    get_fact_ranker,
    retrieve_fact_evidence,
    format_fact_knowledge,
)
from src.pipeline.evidence import (
    retrieve_from_clean_pool,
    prefetch_query_context,
    build_evidence_bundle,
    assess_with_llm,
)
from src.pipeline.selection import split_clean_noisy, finalize_remaining_noisy_with_slm
from src.pipeline.finetune import maybe_finetune_slm_on_clean
from src.slm.dataset import load_data_from_csv
from src.slm.model import IntegratedSLM
from src.llm.handler import get_llm
from src.evaluation.metrics import evaluate_and_plot, compare_models

set_seed(42)

raw_df = pd.read_csv(REPO_ROOT / 'dataset' / 'twitter15_16.csv')
pre_df = pd.read_csv(REPO_ROOT / 'dataset' / 'twitter15_16_preprocessed.csv')

sample_text = str(pre_df.iloc[0].get('text', ''))
if not sample_text.strip():
    sample_text = str(raw_df.iloc[0].get('content', ''))

print('REPO_ROOT:', REPO_ROOT)
print('KNOWLEDGE_MODE:', KNOWLEDGE_MODE)
print('MODEL_PATH:', MODEL_PATH)
print('LLM_MODEL_NAME:', LLM_MODEL_NAME)
print('sample_text preview:', sample_text[:220])


def show_actual_expected(name, actual, expected):
    print(f'[{name}]')
    print('actual  :', actual)
    print('expected:', expected)


def detect_timestamp_col(df):
    for c in ['timestamp', 'created_at', 'time', 'date', 'createdAt']:
        if c in df.columns:
            return c
    return None

## src/utils.py

### Function: set_seed
Expected: random outputs lap lai khi set cung seed.

In [ ]:
set_seed(42)
a1 = [random.random() for _ in range(3)]
set_seed(42)
a2 = [random.random() for _ in range(3)]
show_actual_expected('set_seed', a1, 'same values as second run')
print('equal:', a1 == a2)

### Function: preprocess_text
Expected: lower-case, bo url/mention, chuan hoa khoang trang.

In [ ]:
raw = 'BREAKING!!! Visit https://abc.com NOW @user #Alert'
out = preprocess_text(raw)
show_actual_expected('preprocess_text', out, 'breaking visit now #alert')

### Function: clean_query
Expected: bo dau cau, chuan hoa unicode va khoang trang.

In [ ]:
q = 'COVID-19!!!   vaccine???   '
out = clean_query(q)
show_actual_expected('clean_query', out, 'COVID 19 vaccine')

### Function: truncate_text
Expected: cat chuoi theo max_length va them ... neu vuot gioi han.

In [ ]:
text = 'this is a very long sentence for truncate text function testing'
out = truncate_text(text, max_length=20)
show_actual_expected('truncate_text', out, 'this is a very long... (or similar word-boundary cut)')

## src/labels.py

### Function: generate_demo_label
Expected: tra ve 1 nhan thuoc tap synonym.

In [ ]:
label = generate_demo_label('some demo text')
show_actual_expected('generate_demo_label', label, 'one of REAL/FAKE synonym labels')

### Function: to_clean_demo_label
Expected: 0 -> Real, 1 -> Fake.

In [ ]:
a = to_clean_demo_label(0)
b = to_clean_demo_label(1)
show_actual_expected('to_clean_demo_label(0)', a, 'Real')
show_actual_expected('to_clean_demo_label(1)', b, 'Fake')

### Function: parse_llm_label
Expected: parse duoc response ve nhan 0/1.

In [ ]:
r1 = parse_llm_label('Real')
r2 = parse_llm_label('Fake')
r3 = parse_llm_label('```json Fake ```')
show_actual_expected('parse_llm_label Real', r1, 0)
show_actual_expected('parse_llm_label Fake', r2, 1)
show_actual_expected('parse_llm_label fenced Fake', r3, 1)

## src/prompts.py

### Function: build_dual_extraction_prompt
Expected: prompt co schema entities + query.

In [ ]:
p = build_dual_extraction_prompt(sample_text)
show_actual_expected('build_dual_extraction_prompt contains entities', '"entities"' in p, True)
show_actual_expected('build_dual_extraction_prompt contains query', '"query"' in p, True)
print(p[:400])

### Function: build_entity_extraction_prompt
Expected: prompt chi yeu cau entities.

In [ ]:
p = build_entity_extraction_prompt(sample_text)
show_actual_expected('build_entity_extraction_prompt has entities schema', '"entities"' in p, True)
show_actual_expected('build_entity_extraction_prompt no query schema', '"query"' in p, False)
print(p[:300])

### Function: build_classification_prompt
Expected: prompt gom knowledge, demos va target article.

In [ ]:
demos = [{'text': 'Example text one', 'label': 'Real'}, {'text': 'Example text two', 'label': 'Fake'}]
knowledge = '<ENTITY_DEFINITIONS>demo</ENTITY_DEFINITIONS>'
p = build_classification_prompt(sample_text, knowledge, demos)
show_actual_expected('has BACKGROUND KNOWLEDGE', 'BACKGROUND KNOWLEDGE' in p, True)
show_actual_expected('has EXAMPLES', '[Example 1]' in p, True)
show_actual_expected('has TARGET ARTICLE', 'TARGET ARTICLE TO CLASSIFY' in p, True)
print(p[:450])

## src/retrieval/demo_retrieval.py

### Function: load_news_corpus
Expected: tra ve list van ban tu AG News.

In [ ]:
corpus = load_news_corpus()
show_actual_expected('load_news_corpus type', isinstance(corpus, list), True)
show_actual_expected('load_news_corpus size > 0', len(corpus) > 0, True)
print('first item preview:', str(corpus[0])[:180] if corpus else 'N/A')

### Function: search_news
Expected: tra ve danh sach tin tu Bing backend.

In [ ]:
news = search_news(sample_text, max_results=5)
show_actual_expected('search_news type', isinstance(news, list), True)
show_actual_expected('search_news len <= 5', len(news) <= 5, True)
print('result count:', len(news))
print('first news preview:', str(news[0])[:180] if news else 'N/A')

### Function: retrieve_demonstrations
Expected: BM25 tra ve top-k demo co text/label/source.

In [ ]:
base_corpus = corpus[:300] if corpus else []
demos = retrieve_demonstrations(preprocess_text(sample_text), base_corpus, k=4)
show_actual_expected('retrieve_demonstrations len <= 4', len(demos) <= 4, True)
if demos:
    show_actual_expected('demo keys', sorted(list(demos[0].keys())), ['label', 'source', 'text'])
    print('demo0 preview:', str(demos[0])[:220])

## src/retrieval/knowledge_agent.py (Wiki + Combined)

### Function: query_wikipedia
Expected: tra summary hoac Not found.

In [ ]:
out = query_wikipedia('Barack Obama')
show_actual_expected('query_wikipedia type', isinstance(out, str), True)
show_actual_expected('query_wikipedia non-empty', len(out) > 0, True)
print(out[:220])

### Function: extract_wiki_knowledge_from_entities
Expected: dict entity -> definition.

In [ ]:
wiki_map = extract_wiki_knowledge_from_entities(['Barack Obama', 'NASA'])
show_actual_expected('extract_wiki_knowledge_from_entities type', isinstance(wiki_map, dict), True)
print('keys:', list(wiki_map.keys())[:5])

### Function: format_entity_definitions
Expected: format text co - Entity va - Definition.

In [ ]:
formatted = format_entity_definitions({'NASA': 'US space agency'})
show_actual_expected('contains Entity', '- Entity:' in formatted, True)
show_actual_expected('contains Definition', '- Definition:' in formatted, True)
print(formatted)

### Function: format_verified_reports
Expected: format list chunk thanh report text.

In [ ]:
chunks = [{'title': 'Reuters', 'chunk_text': 'This claim was disproven.'}]
formatted = format_verified_reports(chunks)
show_actual_expected('contains Title', '- Title:' in formatted, True)
show_actual_expected('contains Key Information', '- Key Information:' in formatted, True)
print(formatted)

### Function: build_knowledge_wiki_only
Expected: tra ve combined_text + mode=wiki_only.

In [ ]:
bundle = build_knowledge_wiki_only(sample_text, wiki_fetch_full=False)
show_actual_expected('mode', bundle.get('mode'), 'wiki_only')
show_actual_expected('has combined_text', 'combined_text' in bundle, True)
print(bundle.get('combined_text', '')[:300])

### Function: build_knowledge_full
Expected: tra ve combined_text + mode=full (co the rong neu internet unavailable).

In [ ]:
bundle = build_knowledge_full(sample_text, fact_top_k=2, wiki_fetch_full=False)
show_actual_expected('mode', bundle.get('mode'), 'full')
show_actual_expected('has combined_text', 'combined_text' in bundle, True)
print(bundle.get('combined_text', '')[:300])

### Function: build_knowledge_bundle
Expected: dispatcher tra dung mode theo tham so.

In [ ]:
a = build_knowledge_bundle(sample_text, fact_top_k=2, mode='wiki_only')
b = build_knowledge_bundle(sample_text, fact_top_k=2, mode='full')
show_actual_expected('wiki_only mode', a.get('mode'), 'wiki_only')
show_actual_expected('full mode', b.get('mode'), 'full')

### Function: get_cached_knowledge_bundle_local
Expected: lan goi thu 2 lay tu cache key giong nhau.

In [ ]:
cache = {}
b1 = get_cached_knowledge_bundle_local(sample_text, cache, fact_top_k=2, mode='wiki_only')
b2 = get_cached_knowledge_bundle_local(sample_text, cache, fact_top_k=2, mode='wiki_only')
show_actual_expected('cache size >= 1', len(cache) >= 1, True)
show_actual_expected('same object by cache key', b1 == b2, True)

## src/retrieval/knowledge_retrieval.py (Trusted News, non-wiki)
Section rieng cho retrieval fact-check tu trusted domains.

### Function: analyze_claim_entities_and_query
Expected: tra entities va query tu LLM extraction.

In [ ]:
analysis = analyze_claim_entities_and_query(sample_text, mode='full')
show_actual_expected('has entities', 'entities' in analysis, True)
show_actual_expected('has query', 'query' in analysis, True)
print(analysis)

### Function: build_trusted_domain_query
Expected: tra string OR site:domain.

In [ ]:
q = build_trusted_domain_query(['reuters.com', 'apnews.com'])
show_actual_expected('contains site:reuters.com', 'site:reuters.com' in q, True)
show_actual_expected('contains OR', 'OR' in q, True)
print(q)

### Function: scrape_full_article
Expected: tra text article hoac None.

In [ ]:
out = scrape_full_article('https://www.reuters.com')
show_actual_expected('type is str or None', isinstance(out, str) or out is None, True)
print((out or 'None')[:220])

### Function: _crawl_single_result
Expected: normalize 1 search result thanh dict title/url/content hoac None.

In [ ]:
item = {'title': 'Reuters', 'url': 'https://www.reuters.com', 'snippet': 'Reuters homepage snippet'}
out = _crawl_single_result(item)
show_actual_expected('_crawl_single_result type', isinstance(out, dict) or out is None, True)
print(out if out is None else {k: str(v)[:120] for k, v in out.items()})

### Function: crawl_results_parallel
Expected: crawl nhieu result song song.

In [ ]:
items = [
    {'title': 'Reuters', 'url': 'https://www.reuters.com', 'snippet': 'r1'},
    {'title': 'AP', 'url': 'https://apnews.com', 'snippet': 'r2'},
]
outs = crawl_results_parallel(items, max_workers=2)
show_actual_expected('crawl_results_parallel type', isinstance(outs, list), True)
print('count:', len(outs))

### Function: chunk_text_by_sentences
Expected: tach text thanh cac chunk theo max_words.

In [ ]:
t = 'Sentence one. Sentence two is a bit longer. Sentence three is here. Sentence four for chunking.'
chunks = chunk_text_by_sentences(t, max_words=6, overlap_sentences=1)
show_actual_expected('chunk_text_by_sentences type', isinstance(chunks, list), True)
show_actual_expected('chunk count >= 1', len(chunks) >= 1, True)
print(chunks)

### Function: get_fact_ranker
Expected: tra CrossEncoder singleton.

In [ ]:
ranker = get_fact_ranker()
show_actual_expected('ranker loaded', ranker is not None, True)
print(type(ranker))

### Function: retrieve_fact_evidence
Expected: tra analysis + top_chunks (co the rong neu internet unavailable).

In [ ]:
out = retrieve_fact_evidence(sample_text, max_urls=6, top_k_chunks=2)
show_actual_expected('has analysis', 'analysis' in out, True)
show_actual_expected('has top_chunks', 'top_chunks' in out, True)
print('analysis:', out.get('analysis'))
print('top_chunks count:', len(out.get('top_chunks', [])))

### Function: format_fact_knowledge
Expected: format top_chunks thanh string de debug/prompt.

In [ ]:
formatted = format_fact_knowledge([
    {'score': 5.1, 'title': 'Reuters', 'chunk_text': 'Claim is false', 'url': 'https://reuters.com'}
])
show_actual_expected('contains [F1]', '[F1]' in formatted, True)
print(formatted)

## src/pipeline/evidence.py

### Function: prefetch_query_context
Expected: tra context gom knowledge + bing seeds.

In [ ]:
ctx = prefetch_query_context(sample_text, demo_k=TOP_K_DEMOS, fact_top_k=FACT_TOP_K, knowledge_mode='wiki_only')
show_actual_expected('has knowledge_text', 'knowledge_text' in ctx, True)
show_actual_expected('has bing_seed_news', 'bing_seed_news' in ctx, True)
print('knowledge_mode:', ctx.get('knowledge_mode'))
print('seed count:', len(ctx.get('bing_seed_news', [])))

### Function: retrieve_from_clean_pool
Expected: BM25 lay demos tu D_clean va map label Real/Fake.

In [ ]:
clean_pool = [
    {'text': 'NASA confirms moon mission details', 'label': 0},
    {'text': 'Hoax cure spreads online', 'label': 1},
    {'text': 'Reuters reports official numbers', 'label': 0},
]
demos = retrieve_from_clean_pool(sample_text, clean_pool, k=2)
show_actual_expected('retrieve_from_clean_pool type', isinstance(demos, list), True)
print(demos)

### Function: build_evidence_bundle
Expected: tra demos + knowledge_k + retrieval_source theo round.

In [ ]:
static_corpus = (corpus[:100] if 'corpus' in globals() and corpus else ['news sample 1', 'news sample 2'])
ctx = prefetch_query_context(sample_text, demo_k=4, fact_top_k=2, knowledge_mode='wiki_only')
round1 = build_evidence_bundle(sample_text, static_corpus, clean_pool=[], round_id=1, query_context=ctx, demo_k=4)
round2 = build_evidence_bundle(sample_text, static_corpus, clean_pool=clean_pool, round_id=2, query_context=ctx, demo_k=2)
show_actual_expected('round1 source', round1[2], 'external_prefetched')
show_actual_expected('round2 source', round2[2], 'd_clean (if clean_pool matched)')
print('round1 demo count:', len(round1[0]))
print('round2 demo count:', len(round2[0]))

### Function: assess_with_llm
Expected: goi real LLM de lay y_llm + llm_raw.

In [ ]:
llm = get_llm(LLM_MODEL_NAME)
print('Loaded LLM model:', LLM_MODEL_NAME)
print('LLM device:', getattr(getattr(llm, 'model', None), 'device', 'unknown'))

demos = [{'text': 'This is verified by Reuters.', 'label': 'Real'}]
knowledge_k = '<VERIFIED_REPORTS>Reuters fact-check says false claim.</VERIFIED_REPORTS>'
out = assess_with_llm(sample_text, demos, knowledge_k, llm)
show_actual_expected('has y_llm', 'y_llm' in out, True)
show_actual_expected('y_llm in [0,1]', out.get('y_llm') in [0, 1], True)
print('llm_raw:', out.get('llm_raw'))

## src/pipeline/selection.py

### Function: split_clean_noisy
Expected: True khi LLM-SLM agreement va confidence dat nguong.

In [ ]:
s1 = {'label_llm': 1, 'label_slm': 1, 'conf_slm': 0.95}
s2 = {'label_llm': 0, 'label_slm': 1, 'conf_slm': 0.99}
show_actual_expected('split_clean_noisy positive', split_clean_noisy(s1, 0.8), True)
show_actual_expected('split_clean_noisy negative', split_clean_noisy(s2, 0.8), False)

### Function: finalize_remaining_noisy_with_slm
Expected: gan label_final bang output SLM cho tat ca mau noisy.

In [ ]:
slm = IntegratedSLM(model_path=MODEL_PATH)
print('SLM device:', slm.device)

noisy = [
    {'text': 'Breaking miracle cure with no source'},
    {'text': 'Government issued official statement today'},
]
finalized = finalize_remaining_noisy_with_slm(noisy, slm)
show_actual_expected('finalized type', isinstance(finalized, list), True)
show_actual_expected('all have status finalized_by_slm', all(i.get('status') == 'finalized_by_slm' for i in finalized), True)
print(finalized)

## src/pipeline/finetune.py

### Function: maybe_finetune_slm_on_clean
Expected: skip neu mau it; train khi du so luong va enable.

In [ ]:
small_clean = [{'text': 'sample', 'label': 0} for _ in range(2)]
res_skip = maybe_finetune_slm_on_clean(slm, small_clean, round_id=2, slm_finetune_min_samples=16)
show_actual_expected('skip trained', res_skip.get('trained'), False)
show_actual_expected('skip reason', res_skip.get('reason'), 'insufficient_samples')

## src/slm/dataset.py

### Function: load_data_from_csv
Expected: load train+val merged va test tu csv config.

In [ ]:
train_texts, train_labels, test_texts, test_labels = load_data_from_csv(TRAIN_CSV, VAL_CSV, TEST_CSV)
show_actual_expected('train_texts type', isinstance(train_texts, list), True)
show_actual_expected('train_labels type', isinstance(train_labels, list), True)
show_actual_expected('test_texts type', isinstance(test_texts, list), True)
show_actual_expected('len train > 0', len(train_texts) > 0, True)
print('train size:', len(train_texts), 'test size:', len(test_texts))

## src/slm/model.py (Real SLM)
SLM = RoBERTa fine-tuning voi split train 60% theo thoi gian som nhat.

### Class/Methods: IntegratedSLM, fnetune, inference, inference_batch
Expected: khoi tao model that, train voi hyperparams notebook v1, suy luan single va batch.

In [ ]:
def time_split_60(df, text_col='text', label_col='label_id'):
    ts_col = detect_timestamp_col(df)
    if ts_col is not None:
        ts = pd.to_datetime(pd.to_numeric(df[ts_col], errors='coerce'), unit='s', errors='coerce')
        if ts.isna().mean() > 0.2:
            ts = pd.to_datetime(df[ts_col], errors='coerce')
        work = df.copy()
        work['_ts'] = ts
        work = work.dropna(subset=['_ts', text_col, label_col]).sort_values('_ts').reset_index(drop=True)
    else:
        work = df.dropna(subset=[text_col, label_col]).reset_index(drop=True)

    split_idx = int(len(work) * 0.6)
    train_df = work.iloc[:split_idx].copy()
    holdout_df = work.iloc[split_idx:].copy()

    return (
        train_df[text_col].astype(str).tolist(),
        train_df[label_col].astype(int).tolist(),
        holdout_df[text_col].astype(str).tolist(),
        holdout_df[label_col].astype(int).tolist(),
    )

train60_texts, train60_labels, holdout_texts, holdout_labels = time_split_60(pre_df)
print('train60:', len(train60_texts), 'holdout:', len(holdout_texts))

slm_round = IntegratedSLM(model_path=MODEL_PATH)
train_result = slm_round.fnetune(
    train_texts=train60_texts,
    train_labels=train60_labels,
    model_init='roberta-base',
    epochs=4,
    batch_size=32,
    lr=1e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    save_path='/kaggle/working/mrcd_roberta_finetuned_v1_style',
)
show_actual_expected('fnetune trained', train_result.get('trained'), True)
print(train_result)

single_pred = slm_round.inference(holdout_texts[0])
batch_pred = slm_round.inference_batch(holdout_texts[:8], batch_size=4)
show_actual_expected('inference tuple len', len(single_pred), 3)
show_actual_expected('inference_batch len', len(batch_pred), min(8, len(holdout_texts)))
print('single_pred:', single_pred)

## src/evaluation/metrics.py

### Function: evaluate_and_plot
Expected: tra dict metrics va ve confusion matrix.

In [ ]:
baseline_preds = [slm_round.inference(t)[0] for t in holdout_texts]
baseline_metrics = evaluate_and_plot(
    y_true=holdout_labels,
    y_pred=baseline_preds,
    labels=['Real', 'Fake'],
    model_name='SLM Baseline (after 60pct early-time train)'
)
show_actual_expected('evaluate_and_plot has accuracy', 'accuracy' in baseline_metrics, True)
print('baseline accuracy:', baseline_metrics['accuracy'])

## MRCD Finetune Simulation (Round 2 / Round 3)
Mo phong theo yeu cau: baseline tren test, fine-tune +50 mau round2, tiep +50 mau round3, roi so sanh.

In [ ]:
# Tao index va chia 2 batch 50 tu holdout
n = len(holdout_texts)
idx = np.arange(n)
np.random.seed(42)
np.random.shuffle(idx)

r2_idx = idx[:min(50, n)]
r3_idx = idx[min(50, n):min(100, n)]

def build_clean_pool_from_indices(indices):
    return [{'text': holdout_texts[i], 'label': int(holdout_labels[i])} for i in indices]

r2_clean = build_clean_pool_from_indices(r2_idx)
r3_clean = build_clean_pool_from_indices(r3_idx)

# Round 2 finetune
res_r2 = maybe_finetune_slm_on_clean(
    slm_round,
    r2_clean,
    round_id=2,
    enable_slm_finetune=True,
    slm_finetune_epochs=1,
    slm_finetune_batch_size=32,
    slm_finetune_lr=1e-5,
    slm_finetune_weight_decay=1e-4,
    slm_finetune_min_samples=16,
)
print('round2 finetune:', res_r2)

pred_r2 = [slm_round.inference(t)[0] for t in holdout_texts]
metrics_r2 = evaluate_and_plot(
    holdout_labels,
    pred_r2,
    labels=['Real', 'Fake'],
    model_name='SLM after Round 2 (+50)'
)

# Round 3 finetune
res_r3 = maybe_finetune_slm_on_clean(
    slm_round,
    r3_clean,
    round_id=3,
    enable_slm_finetune=True,
    slm_finetune_epochs=1,
    slm_finetune_batch_size=32,
    slm_finetune_lr=1e-5,
    slm_finetune_weight_decay=1e-4,
    slm_finetune_min_samples=16,
)
print('round3 finetune:', res_r3)

pred_r3 = [slm_round.inference(t)[0] for t in holdout_texts]
metrics_r3 = evaluate_and_plot(
    holdout_labels,
    pred_r3,
    labels=['Real', 'Fake'],
    model_name='SLM after Round 3 (+50)'
)

### Function: compare_models
Expected: bang so sanh Baseline vs Round2 vs Round3.

In [ ]:
comparison_df = compare_models({
    'Baseline': baseline_metrics,
    'Round2_+50': metrics_r2,
    'Round3_+50': metrics_r3,
})
print(comparison_df)